In [1]:
import numpy as np
import pandas as pd

SEED = 42

EXPERIMENT_START = "2026-03-02"
ENROLL_DAYS = 21
OBSERVE_DAYS = 60

# 규모
DAILY_EXPOSURES = 15_000

#작품수
N_COMPLETED_TITLES = 4_800
N_SERIAL_TITLES = 1_500

#장르
GENRES = ("로맨스판타지", "현대판타지","무협","로맨스","기타")
GENRE_SHARE = [0.35, 0.25, 0.15, 0.15, 0.10]

#읽기 패턴
SEGMENTS = ["serial","binge","explorer"] #연재 추격형, 완결 정주행형, 탐색형
SEGMENT_SHARE = [0.45, 0.30, 0.25]

#가격
PASS_PRICE = 9_900
COIN_PRICE = 200

rng = np.random.default_rng(SEED)


In [2]:
n_titles = N_COMPLETED_TITLES + N_SERIAL_TITLES 

titles = pd.DataFrame({
    "title_id" : [f"T{i:05d}" for i  in range(1, n_titles + 1)],
    "genre": rng.choice(GENRES, size = n_titles, p=GENRE_SHARE),
    "status":["completed"] * N_COMPLETED_TITLES + ["serial"] * N_SERIAL_TITLES
})    

# 회차 수 : 완결작 평균 220화, 연재작 평균 90화 (최소 30화)
titles["total_episodes"] = np.where(
    titles["status"] == "completed",
    rng.normal(220, 60, n_titles),
    rng.normal(90, 30, n_titles),
).clip(30).round().astype(int)

titles.groupby("status")["total_episodes"].mean()

status
completed    219.086667
serial        91.163333
Name: total_episodes, dtype: float64

In [3]:
#유저 모집단

# 전체 유저 수 = 하루 노출자 × 모집 일수 (15,000 × 21 = 315,000명)
n_users = DAILY_EXPOSURES * ENROLL_DAYS 

# 문자열 날짜를 pandas 날짜타입으로 변환 
start = pd.Timestamp(EXPERIMENT_START)

# 유저 모집단 테이블 생성
base = pd.DataFrame({
    "user_id" : [f"U{i:06d}" for i in range(1, n_users +1)],  #유저 아이디 
    "segment" : rng.choice(SEGMENTS, size = n_users, p=SEGMENT_SHARE)
})

# 첫 노출 시각 만들기 
#며칠ㄷ째 노출됐는지. 
day = rng.integers(0, ENROLL_DAYS, n_users) 

# 몇 시에 노출됐는지: 0시 ~ 23시
# 시간대별 가중치 → 숫자가 클수록 그 시간에 노출이 많음 (밤 21~23시가 가장 높음)
hour_weights = np.array([3,2,1,1,1,1,1,2,3,3,3,3,4,4,3,3,3,4,5,6,7,8,9,7], dtype=float)
#가중치를 합계 1인 확률로 바꿔서 그 비율대로 시간 선택
hour = rng.choice(24, size=n_users, p=hour_weights / hour_weights.sum())

# 몇분에 노출되었는지 : 0~59분
minute = rng.integers(0,60,n_users)

#시작일 + 일 + 시 + 분을 더해 최종 노출 시각 생성 
base["exposed_at"] = (start 
                      + pd.to_timedelta(day, "D")
                      + pd.to_timedelta(hour, "h")
                      + pd.to_timedelta(minute, "min"))

# --실험군 배정 -- 
# A(7일 체험) , B(14일 체험) 중 50:50 랜덤 배정
#p를 안주면 균등 확률로 선택
base["variant"] = rng.choice(["A","B"], size = n_users)

# -- 확인 -- 
print(base["segment"].value_counts(normalize=True).round(3))
print(base["variant"].value_counts())



segment
serial      0.45
binge       0.30
explorer    0.25
Name: proportion, dtype: float64
variant
A    157500
B    157500
Name: count, dtype: int64


In [4]:
# 4. 세그먼트별 행동 파라미터

# 행 = 세그먼트, 열 = 행동 파라미터
# 도메인 설정서 6장(유저 세그먼트) 수치를 기준으로 설정
seg_params = pd.DataFrame({
    "active_prob" : [0.60, 0.35, 0.30],   # 30일 중 하루에 접속할 확률 (연재 추격형이 가장 자주 접속)
    "daily_reads" : [6, 25, 3],            # 접속한 날 평균 열람 회차 수 (설정서: 6화 / 25화 / 3화)
 
    # 열람 중 연재작 비중: 베타분포 Beta(a, b)로 사람마다 다르게 뽑음
    # 평균 = a / (a + b) → 연재 추격형 0.8, 완결 정주행형 0.2, 탐색형 0.5
    "serial_a" : [8,2,5],
    "serial_b" : [2,8,5],

    # 열람 중 무료 회차(첫 3화) 비중
    # 평균 -> 연재 추격형 0.2, 완결 정주행형 약 0.11, 탐색형 0.7(탐색혁은 무료 위주)
    "free_a": [2, 1.5, 7],
    "free_b": [8, 12, 3],
    
    "pay_prob": [0.45, 0.06, 0.08],         # 30일 안에 코인 결제할 확률
    "spend_mean": [15_000, 20_000, 6_000],  # 결제한 사람의 평균 결제액(원)
}, index=SEGMENTS)

# 유저마다 자기 세그먼트의 파라미터를 한 줄씩 붙임
# .loc[세그먼트 목록] → 유저 순서대로 해당 세그먼트 행을 가져옴
# reset_index(drop=True) → 인덱스를 0, 1, 2 ...로 다시 매겨서 base와 줄 맞춤

p = seg_params.loc[base["segment"]].reset_index(drop=True)
p.head()
p.tail()

,active_prob,daily_reads,serial_a,serial_b,free_a,free_b,pay_prob,spend_mean
314995,0.60,6,8,2,2.0,8,0.45,15000
314996,0.60,6,8,2,2.0,8,0.45,15000
314997,0.60,6,8,2,2.0,8,0.45,15000
314998,0.35,25,2,8,1.5,12,0.06,20000
314999,0.35,25,2,8,1.5,12,0.06,20000


In [5]:
# 5. 가입일 

# == 신규 / 기존 유저 구분 -- 
is_new = rng.random(n_users) < 0.20

#노출일 기준 가입 후 경과일
#신규 : 0~29일 전 가입 / 기존 : 30~1799일 전 가입 (서비스 출시 2021년 이후)

days_since_signup = np.where(
    is_new,
    rng.integers(0,30, n_users),
    rng.integers(30,1800, n_users),
)

#가입일 = 노출일 - 경과일
base["signup_date"] = (base["exposed_at"].dt.normalize() - pd.to_timedelta(days_since_signup, "D")).dt.date 

# 실험 전 30일 중 실제로 가입해 있던 비율
# 가입 10일 된 신규 유저는 30일 중 10일만 기록이 있어야 하므로 10/30 = 0.33
tenure = np.minimum(days_since_signup, 30)/30



In [6]:
# 6. 실험전 30일 열람과 결제
# ── 열람 회차 수 ──

# 30일 중 접속한 날 수: 이항분포 (30번 시도 × 접속 확률)
# tenure를 곱해서 가입 기간이 짧은 유저는 접속일도 줄임
active_days = rng.binomial(30, p["active_prob"]) * tenure

# 총 열람 회차: 포아송분포 (평균 = 접속일 × 하루 평균 열람)
total_reads = rng.poisson(active_days * p["daily_reads"])

# 유저별 연재작 비중, 무료 회차 비중 뽑기
serial_share = rng.beta(p["serial_a"], p["serial_b"])
free_share = rng.beta(p["free_a"], p["free_b"])

# 연재작 열람 = 총 열람 중 연재작 비중만큼 (이항분포로 정수 개수 뽑기)
base["pre30_serial_reads"] = rng.binomial(total_reads, serial_share)
# 완결작 열람 = 나머지
base["pre30_completed_reads"] = total_reads - base["pre30_serial_reads"]
# 무료 회차 열람 = 총 열람 중 무료 비중만큼 (연재·완결 열람에 포함된 부분집합)
base["pre30_free_reads"] = rng.binomial(total_reads, free_share)

# ── 코인 결제액 ──

# 결제 여부: 세그먼트별 결제 확률 × 가입 기간 비율
paid = rng.random(n_users) < p["pay_prob"] * tenure

# 결제액: 감마분포 (0 이상, 오른쪽으로 꼬리가 긴 분포 → 소수가 많이 쓰는 실제 결제 패턴)
# shape=2, scale=평균/2 → 평균이 spend_mean이 됨
spend = rng.gamma(2, p["spend_mean"] / 2)

# 결제 안 한 사람은 0원, 결제한 사람은 100원 단위로 반올림
base["pre30_coin_spend"] = np.where(
    paid,
    (spend / COIN_PRICE).round() * COIN_PRICE,
    0,
).astype(int)




In [7]:
AGE_GROUPS = ["10대", "20대", "30대", "40대 이상"]

# 세그먼트별 연령대 분포 (모두 20~30대 중심)
age_p = {
    "serial":   [0.10, 0.40, 0.32, 0.18],
    "binge":    [0.12, 0.45, 0.30, 0.13],
    "explorer": [0.15, 0.40, 0.27, 0.18],
}

# 세그먼트별 선호 장르 분포 (GENRES 순서: 로판, 현판, 무협, 로맨스, 기타)
genre_p = {
    "serial":   [0.25, 0.40, 0.20, 0.05, 0.10],  # 현대판타지 중심 (연재 추격 성향)
    "binge":    [0.50, 0.10, 0.10, 0.25, 0.05],  # 로맨스판타지·로맨스 중심 (완결작 정주행)
    "explorer": GENRE_SHARE,                     # 전체 평균과 같음
}

# 빈 컬럼을 먼저 만들고, 세그먼트별로 채움
base["age_group"] = None
base["preferred_genre"] = None

for seg in SEGMENTS:
    m = base["segment"] == seg   # 이 세그먼트에 해당하는 행만 True
    base.loc[m, "age_group"] = rng.choice(AGE_GROUPS, size=m.sum(), p=age_p[seg])
    base.loc[m, "preferred_genre"] = rng.choice(GENRES, size=m.sum(), p=genre_p[seg])

# 유입 경로: 신규 유저는 광고 유입 비중이 더 높게
CHANNELS = ["organic", "paid_ad", "sns"]
base["acquisition_channel"] = np.where(
    is_new,
    rng.choice(CHANNELS, size=n_users, p=[0.35, 0.45, 0.20]),  # 신규
    rng.choice(CHANNELS, size=n_users, p=[0.60, 0.26, 0.14]),  # 기존
)

In [8]:
# 최종 users 테이블: 설계서 컬럼만 골라냄
# segment, variant, exposed_at은 빼서 정답이 드러나지 않게 함
users = base[["user_id", "signup_date", "age_group", "acquisition_channel",
              "preferred_genre", "pre30_serial_reads", "pre30_completed_reads",
              "pre30_free_reads", "pre30_coin_spend"]]

# 확인 1: 세그먼트별 평균 (정답 컬럼 segment를 써서 의도대로 만들어졌는지 검증)
print(base.groupby("segment")[["pre30_serial_reads", "pre30_completed_reads",
                               "pre30_free_reads", "pre30_coin_spend"]].mean().round(1))

# 확인 2: 세그먼트별 코인 결제자 비율
print((base["pre30_coin_spend"] > 0).groupby(base["segment"]).mean().round(3))

users.head()

          pre30_serial_reads  pre30_completed_reads  pre30_free_reads  \
segment                                                                 
binge                   47.2                  188.6              26.3   
explorer                12.1                   12.1              16.9   
serial                  77.4                   19.4              19.4   

          pre30_coin_spend  
segment                     
binge               1074.6  
explorer             435.1  
serial              6063.0  
segment
binge       0.055
explorer    0.073
serial      0.405
Name: pre30_coin_spend, dtype: float64


,user_id,signup_date,age_group,acquisition_channel,preferred_genre,pre30_serial_reads,pre30_completed_reads,pre30_free_reads,pre30_coin_spend
0,U000001,2024-09-03,20대,paid_ad,로맨스판타지,52,324,13,0
1,U000002,2025-08-08,20대,organic,로맨스,47,158,14,0
2,U000003,2021-11-08,20대,sns,현대판타지,13,19,26,0
3,U000004,2026-03-17,20대,organic,무협,0,0,0,0
4,U000005,2026-02-27,10대,paid_ad,현대판타지,3,23,0,0


In [9]:
# 9. exposures 테이블 

TRIAL_DAYS = {"A" : 7, "B" : 14}

base["platform"] = rng.choice(["app","web"], size=n_users, p= [0.85, 0.15])

#exposure 테이블
exposures = base[["user_id","variant","exposed_at","platform"]]
exposures.head()


,user_id,variant,exposed_at,platform
0,U000001,A,2026-03-03 19:09:00,app
1,U000002,B,2026-03-03 22:50:00,web
2,U000003,A,2026-03-08 10:06:00,app
3,U000004,B,2026-03-17 23:50:00,app
4,U000005,A,2026-03-06 21:03:00,app


In [10]:
# ── 체험 시작 확률 ──
# 세그먼트별 기본 확률: 완결 정주행형이 가장 높음
# (소개 화면이 유료 완결작 회차를 열 때 뜨기 때문에, 완결작을 많이 읽는 사람이 가장 끌림)
# 가중평균 약 8% (도메인 설정서: 체험 시작률 8%)
start_rate = base["segment"].map({"serial": 0.07, "binge": 0.12, "explorer": 0.045})

# B그룹은 "14일 무료" 문구가 더 매력적이라 체험 시작 확률 12% 상승
# → 실험 설계서에서 걱정했던 '선택 편향'을 일부러 심어둔 부분
start_lift = np.where(base["variant"] == "B", 1.12, 1.0)

# 난수가 (기본 확률 × 상승 배수)보다 작으면 체험 시작
base["started"] = rng.random(n_users) < start_rate * start_lift

# ── 체험 시작 시각 ──
# 노출 후 평균 10분 뒤 시작 (지수분포: 대부분 바로 누르고, 일부만 한참 뒤에 누르는 패턴)
delay_min = rng.exponential(10, n_users)
# .dt.floor("s") → 초 단위 아래 소수점 버림
# 체험 안 한 유저에게도 값이 생기지만, 아래에서 체험 시작자만 골라 쓰므로 상관없음
base["trial_start_at"] = (base["exposed_at"] + pd.to_timedelta(delay_min, "min")).dt.floor("s")

# 확인: 그룹별 체험 시작률 (A 약 7.9%, B 약 8.8%)
base.groupby("variant")["started"].mean().round(4)

variant
A    0.0785
B    0.0879
Name: started, dtype: float64

In [11]:
# 11.체험 중 읽기와 완독 여부 
# ── 체험 시작자만 따로 떼어냄 ──
# .copy() → 원본 base와 분리 (t를 고쳐도 base는 안 바뀜)
t = base[base["started"]].copy()
n_t = len(t)

# 체험 기간과 종료 시각
t["trial_days"] = t["variant"].map(TRIAL_DAYS)                                 # 7 또는 14
t["trial_end_at"] = t["trial_start_at"] + pd.to_timedelta(t["trial_days"], "D")

# ── 읽는 속도 ──
# 개인별 몰입도: 평균 1인 감마분포 (0.5면 평균의 절반, 2면 두 배로 읽는 사람)
t["intensity"] = rng.gamma(4, 1/4, n_t)
# 하루 열람 회차 = 세그먼트 평균 × 개인 몰입도
t["daily_rate"] = t["segment"].map({"serial": 6, "binge": 25, "explorer": 3}) * t["intensity"]
# 체험 중 접속 확률 (체험 중에는 평소보다 자주 들어옴)
t["active_p"] = t["segment"].map({"serial": 0.85, "binge": 0.80, "explorer": 0.40})

# ── 체험 중 주로 읽는 완결작 1편 ──
completed_ids = titles.loc[titles["status"] == "completed", "title_id"].to_numpy()
t["main_title"] = rng.choice(completed_ids, n_t)
# 그 작품의 총 회차 수 붙이기 (title_id → total_episodes 변환표로 매핑)
eps_map = titles.set_index("title_id")["total_episodes"]
t["main_title_eps"] = t["main_title"].map(eps_map)

# ── 완독까지 걸리는 일수 ──
# 완결 정주행형은 읽는 양의 90%를 완결작에, 나머지는 20%만 완결작에 씀
completed_share = np.where(t["segment"] == "binge", 0.9, 0.2)
# 걸리는 일수 = 작품 회차 ÷ (하루 완결작 열람량 × 접속 확률), 올림
t["days_to_finish"] = np.ceil(
    t["main_title_eps"] / (t["daily_rate"] * completed_share * t["active_p"])
).astype(int)

# 체험 기간 안에 다 읽었는지
t["finished_in_trial"] = t["days_to_finish"] <= t["trial_days"]

# 확인: 완결 정주행형 완독률 A 약 14%, B 약 55% (나머지 세그먼트는 0%)
t.groupby(["segment", "variant"])["finished_in_trial"].mean().round(3).unstack()

variant,A,B
segment,,
binge,0.136,0.545
explorer,0.000,0.000
serial,0.000,0.000


In [12]:
# 12 유료 전환여부
# ── 세그먼트 × 그룹별 기본 전환율 ──
# (세그먼트, 그룹) 짝을 키로 하는 딕셔너리
conv_rate = {("serial", "A"): 0.31, ("serial", "B"): 0.38,      # 연재 추격형: 14일이 유리
             ("binge", "A"): 0.37, ("binge", "B"): 0.37,        # 완결 정주행형: 기본은 같고, 완독 여부로 갈림
             ("explorer", "A"): 0.15, ("explorer", "B"): 0.16}  # 탐색형: 거의 차이 없음

# 유저마다 자기 (세그먼트, 그룹)에 맞는 전환율 가져오기
# zip → 두 컬럼을 한 줄씩 짝지어서 꺼냄
p_conv = np.array([conv_rate[(s, v)] for s, v in zip(t["segment"], t["variant"])])

# 많이 읽은 사람일수록 전환 확률 상승 (몰입도의 제곱근만큼, 0.5~1.6 범위로 제한)
p_conv = p_conv * np.clip(t["intensity"], 0.5, 1.6) ** 0.5

# 체험 중 완독한 사람은 전환 확률이 55% 수준으로 떨어짐 (읽을 거 다 읽고 떠남)
p_conv = np.where(t["finished_in_trial"], p_conv * 0.55, p_conv)

# 전환 여부 결정
t["converted"] = rng.random(n_t) < p_conv

# 확인 1: 세그먼트 × 그룹별 전환율
print(t.groupby(["segment", "variant"])["converted"].mean().round(3).unstack())
# 확인 2: 전체 그룹별 전환율 (A 약 29.2%, B 약 28.8%)
print(t.groupby("variant")["converted"].mean().round(4))

variant       A      B
segment               
binge     0.335  0.261
explorer  0.145  0.163
serial    0.297  0.363
variant
A    0.2922
B    0.2880
Name: converted, dtype: float64


In [13]:
#13.해지여부
# ── 첫 달 해지율 (2회차 결제 전 해지) ──
# 연재 추격형 B는 습관이 생겨 해지율 낮음, 완결 정주행형 B는 오히려 높음
churn_rate = {("serial", "A"): 0.20, ("serial", "B"): 0.17,
              ("binge", "A"): 0.30, ("binge", "B"): 0.33,
              ("explorer", "A"): 0.35, ("explorer", "B"): 0.35}
p_churn = np.array([churn_rate[(s, v)] for s, v in zip(t["segment"], t["variant"])])

# 체험 중 완독한 사람은 해지 확률 10%p 추가 (전환해도 금방 떠남)
p_churn = np.where(t["finished_in_trial"], p_churn + 0.10, p_churn)

# 첫 달 해지: 전환한 사람 중에서만
t["churn_m1"] = t["converted"] & (rng.random(n_t) < p_churn)
# 둘째 달 해지: 첫 달에 안 떠난 사람 중 15%
t["churn_m2"] = t["converted"] & ~t["churn_m1"] & (rng.random(n_t) < 0.15)

# 확인: 유료 전환자 중 첫 달 해지율 (A 약 27%, B 약 28%)
t[t["converted"]].groupby("variant")["churn_m1"].mean().round(3)

variant
A    0.269
B    0.283
Name: churn_m1, dtype: float64

In [14]:
#14. subscription_events 
# ── 이벤트 행을 만드는 함수 ──
# 유저 목록, 이벤트 종류, 발생 시각, 금액을 받아서 이벤트 테이블 조각을 만듦
def make_events(df, event_type, event_at, amount):
    return pd.DataFrame({"user_id": df["user_id"], "event_type": event_type,
                         "event_at": event_at, "amount_krw": amount})

# ── 유저 그룹 나누기 ──
conv = t[t["converted"]]                       # 유료 전환자
not_conv = t[~t["converted"]]                  # 체험만 하고 떠난 사람 (~ 는 '아니다')
churn1 = t[t["churn_m1"]]                      # 첫 달 해지자
stay1 = t[t["converted"] & ~t["churn_m1"]]     # 첫 달 유지 → 2회차 결제
churn2 = t[t["churn_m2"]]                      # 둘째 달 해지자

# ── 해지 시각 ──
# 각 기간의 5~95% 사이 아무 시점에 해지 (기간 맨 처음·맨 끝은 피함)
cancel_in_trial = not_conv["trial_start_at"] + pd.to_timedelta(
    not_conv["trial_days"] * rng.uniform(0.05, 0.95, len(not_conv)), "D")
cancel_m1 = churn1["trial_end_at"] + pd.to_timedelta(
    30 * rng.uniform(0.05, 0.95, len(churn1)), "D")
cancel_m2 = churn2["trial_end_at"] + pd.to_timedelta(
    30 + 30 * rng.uniform(0.05, 0.95, len(churn2)), "D")

# ── 이벤트 조각들을 위아래로 이어 붙이기 ──
events = pd.concat([
    make_events(t, "trial_start", t["trial_start_at"], 0),                         # 모든 체험 시작자
    make_events(not_conv, "trial_cancel", cancel_in_trial.dt.floor("s"), 0),       # 체험 중 해지
    make_events(conv, "payment", conv["trial_end_at"], PASS_PRICE),                # 1회차 결제 (체험 종료 시점)
    make_events(churn1, "cancel", cancel_m1.dt.floor("s"), 0),                     # 첫 달 해지
    make_events(stay1, "payment", stay1["trial_end_at"] + pd.Timedelta(days=30), PASS_PRICE),  # 2회차 결제
    make_events(churn2, "cancel", cancel_m2.dt.floor("s"), 0),                     # 둘째 달 해지
], ignore_index=True)

# ── 관찰 기간(체험 시작 후 60일) 밖 이벤트 제거 ──
# 각 이벤트의 유저 체험 시작 시각을 찾아서 + 60일
observe_end = (events["user_id"].map(t.set_index("user_id")["trial_start_at"])
               + pd.Timedelta(days=OBSERVE_DAYS))
events = events[events["event_at"] <= observe_end]

# 시간순 정렬 후 이벤트 ID 부여 (E0000001 ...)
events = events.sort_values(["event_at", "user_id"]).reset_index(drop=True)
events.insert(0, "event_id", [f"E{i:07d}" for i in range(1, len(events) + 1)])
subscription_events = events

# 확인: 이벤트 종류별 건수
print(subscription_events["event_type"].value_counts())
subscription_events.head()

event_type
trial_start     26206
trial_cancel    18607
payment         13097
cancel           2635
Name: count, dtype: int64


,event_id,user_id,event_type,event_at,amount_krw
0,E0000001,U088741,trial_start,2026-03-02 00:00:43,0
1,E0000002,U082792,trial_start,2026-03-02 00:03:41,0
2,E0000003,U231636,trial_start,2026-03-02 00:06:12,0
3,E0000004,U212177,trial_start,2026-03-02 00:07:28,0
4,E0000005,U256096,trial_start,2026-03-02 00:09:08,0


In [15]:
#15. 유저별 패스 이용 일 수 

# ── 체험 시작 후 며칠 동안 패스가 켜져 있었나 ──
# np.select(조건 목록, 값 목록, default) → 위에서부터 처음 맞는 조건의 값을 사용
t["pass_days"] = np.select(
    [~t["converted"],          # 전환 안 함 → 체험 기간만
     t["churn_m1"],            # 첫 달 해지 → 체험 + 30일
     t["churn_m2"]],           # 둘째 달 해지 → 체험 + 60일
    [t["trial_days"],
     t["trial_days"] + 30,
     t["trial_days"] + 60],
    default=OBSERVE_DAYS,      # 계속 유지 → 관찰 기간 60일 내내
).clip(max=OBSERVE_DAYS)       # 관찰 기간 60일을 넘는 값은 60으로 자름

# 확인: 전환 안 한 사람은 A 7일, B 14일
t.groupby(["variant", "converted"])["pass_days"].mean()

variant  converted
A        False         7.000000
         True         53.816999
B        False        14.000000
         True         55.465262
Name: pass_days, dtype: float64

In [16]:
#16. 유저 X 날짜 격자와 열람 회차
# ── 유저 × 60일 격자 만들기 ──
days = np.arange(OBSERVE_DAYS)                          # [0, 1, 2, ..., 59]
grid = pd.DataFrame({
    # np.repeat → 유저 ID를 60번씩 반복: U1, U1, ..., U1, U2, U2, ...
    "user_id": np.repeat(t["user_id"].to_numpy(), OBSERVE_DAYS),
    # np.tile → 0~59를 유저 수만큼 통째로 반복: 0, 1, ..., 59, 0, 1, ...
    "day": np.tile(days, n_t),
})

# 유저 정보 붙이기 (user_id 기준 결합)
info = t.set_index("user_id")[["segment", "trial_start_at", "daily_rate", "active_p",
                               "pass_days", "days_to_finish", "main_title"]]
grid = grid.join(info, on="user_id")

# 실제 날짜 = 체험 시작일 + day
grid["read_date"] = (grid["trial_start_at"].dt.normalize()
                     + pd.to_timedelta(grid["day"], "D")).dt.date

# 오늘 패스가 켜져 있나
grid["on_pass"] = grid["day"] < grid["pass_days"]

# ── 오늘 접속했나 ──
# 패스 중: 셀 11에서 정한 체험 중 접속 확률 / 패스 없을 때: 평소 접속 확률 (users 만들 때와 같은 값)
off_active = grid["segment"].map({"serial": 0.60, "binge": 0.35, "explorer": 0.30})
grid["active"] = rng.random(len(grid)) < np.where(grid["on_pass"], grid["active_p"], off_active)

# ── 오늘 몇 화 읽었나 ──
rate = grid["daily_rate"].to_numpy()

# 완결 정주행형이 완독한 날 이후로는 읽을 거리가 줄어 열람량 절반
after_finish = (grid["segment"] == "binge") & (grid["day"] >= grid["days_to_finish"])
rate = np.where(after_finish, rate * 0.5, rate)

# 완결 정주행형은 패스가 없으면 회차마다 돈을 내야 해서 열람량 40%로 감소
rate = np.where(~grid["on_pass"] & (grid["segment"] == "binge"), rate * 0.4, rate)

# 접속한 날만 포아송분포로 열람 회차 뽑기, 접속 안 한 날은 0
reads = np.where(grid["active"], rng.poisson(rate), 0)

# 연재작 / 완결작 나누기 (완결작 비중: 연재 추격형 20%, 완결 정주행형 90%, 탐색형 50%)
c_share = grid["segment"].map({"serial": 0.2, "binge": 0.9, "explorer": 0.5}).to_numpy()
grid["completed_reads"] = rng.binomial(reads, c_share)
grid["serial_reads"] = reads - grid["completed_reads"]

# 완독한 날 표시: 완결 정주행형이 패스로 읽다가 마지막 회차에 도달한 날 = 1
grid["titles_finished"] = ((grid["segment"] == "binge")
                           & (grid["day"] == grid["days_to_finish"] - 1)
                           & grid["on_pass"]).astype(int)

# 하나도 안 읽은 날은 버림 (열람 기록이 있는 날만 남김)
grid = grid[reads > 0].copy()
len(grid)

872920

In [17]:
#17. 이용방식별로 나누기 
# ── 연재작 회차를 무엇으로 열었나 ──
sr = grid["serial_reads"].to_numpy()

# 최신화 코인 결제: 연재작 열람의 25% (최신 10화는 패스로도 못 열어서 패스 유무와 무관)
latest = rng.binomial(sr, 0.25)

# 대기 건너뛰기 코인 결제: 패스 중 5%, 패스 없을 때 15%
# → 패스의 '대기 3시간 단축' 때문에 코인을 덜 씀 = 잠식효과
skip_p = np.where(grid["on_pass"], 0.05, 0.15)
skip = rng.binomial(sr - latest, skip_p)

# 나머지는 기다리면 무료로 열람
serial_wait_free = sr - latest - skip

# ── 완결작 회차를 무엇으로 열었나 ──
cr = grid["completed_reads"].to_numpy()
on = grid["on_pass"].to_numpy()

# 패스 없을 때: 40%는 무료 회차(첫 3화 등), 60%는 코인 결제
comp_free_off = rng.binomial(cr, 0.4)
comp_coin_off = cr - comp_free_off

# 패스 중이면 완결작 전부 패스로 열람 (코인 0)
comp_pass = np.where(on, cr, 0)
comp_free = np.where(on, 0, comp_free_off)
comp_coin = np.where(on, 0, comp_coin_off)

# 코인 결제 테이블에서 쓸 값 저장
grid["latest"] = latest
grid["skip"] = skip
grid["comp_coin"] = comp_coin

# ── 이용 방식별로 행을 나눠서 이어 붙이기 ──
zeros = np.zeros_like(sr)     # 0으로 채운 배열 (해당 열람이 없는 칸용)
parts = []
for access_type, s_reads, c_reads in [
    ("free",      zeros,            comp_free),   # 무료: 완결작 무료 회차
    ("wait_free", serial_wait_free, zeros),       # 기다리면 무료: 연재작
    ("coin",      latest + skip,    comp_coin),   # 코인: 연재 최신화·대기 건너뛰기 + 완결작
    ("pass",      zeros,            comp_pass),   # 패스: 완결작
]:
    part = pd.DataFrame({
        "user_id": grid["user_id"].to_numpy(),
        "read_date": grid["read_date"].to_numpy(),
        "access_type": access_type,
        "serial_reads": s_reads,
        "completed_reads": c_reads,
        # 완독 표시는 패스 행에만 남김 (패스로 읽다가 완독했으므로)
        "titles_finished": grid["titles_finished"].to_numpy() if access_type == "pass" else 0,
    })
    # 열람이 0인 행은 버림
    parts.append(part[(part["serial_reads"] + part["completed_reads"]) > 0])

daily_reading = (pd.concat(parts, ignore_index=True)
                 .sort_values(["user_id", "read_date", "access_type"])
                 .reset_index(drop=True))

print(len(daily_reading))
daily_reading.head()

1864928


,user_id,read_date,access_type,serial_reads,completed_reads,titles_finished
0,U000014,2026-03-21,coin,1,0,0
1,U000014,2026-03-21,pass,0,1,0
2,U000014,2026-03-21,wait_free,1,0,0
3,U000014,2026-03-23,pass,0,1,0
4,U000014,2026-03-23,wait_free,1,0,0


In [18]:
#18. 코인결제
serial_ids = titles.loc[titles["status"] == "serial", "title_id"].to_numpy()
hour_p = hour_weights / hour_weights.sum()     # 셀 3에서 만든 시간대 가중치 재사용

coin_parts = []
for purpose, coins, title_source in [
    ("latest_episode",    grid["latest"].to_numpy(),    "serial"),   # 연재작 최신화
    ("skip_wait",         grid["skip"].to_numpy(),      "serial"),   # 대기 건너뛰기
    ("completed_episode", grid["comp_coin"].to_numpy(), "main"),     # 완결작 회차
]:
    m = coins > 0              # 코인을 쓴 날만
    g = grid[m]
    n = m.sum()

    # 어떤 작품에 썼나: 연재작은 아무 연재작, 완결작은 체험 때 읽던 작품
    title = rng.choice(serial_ids, n) if title_source == "serial" else g["main_title"].to_numpy()

    # 결제 시각: 그날의 0시 + 시간대 가중치로 뽑은 시각 + 0~3599초
    hour = rng.choice(24, size=n, p=hour_p)
    txn_at = (pd.to_datetime(g["read_date"])
              + pd.to_timedelta(hour, "h")
              + pd.to_timedelta(rng.integers(0, 3600, n), "s"))

    coin_parts.append(pd.DataFrame({
        "user_id": g["user_id"].to_numpy(),
        "title_id": title,
        "txn_at": txn_at.to_numpy(),
        "coins": coins[m],
        "purpose": purpose,
    }))

# 시간순 정렬 후 거래 ID 부여 (C0000001 ...)
coin_transactions = (pd.concat(coin_parts, ignore_index=True)
                     .sort_values(["txn_at", "user_id"])
                     .reset_index(drop=True))
coin_transactions.insert(0, "txn_id", [f"C{i:07d}" for i in range(1, len(coin_transactions) + 1)])

print(len(coin_transactions))
coin_transactions.head()

825352


,txn_id,user_id,title_id,txn_at,coins,purpose
0,C0000001,U200640,T04906,2026-03-02 00:00:27,1,skip_wait
1,C0000002,U258383,T05442,2026-03-02 00:01:08,1,latest_episode
2,C0000003,U178529,T05255,2026-03-02 00:02:39,4,latest_episode
3,C0000004,U190504,T05921,2026-03-02 00:02:52,1,latest_episode
4,C0000005,U266425,T05405,2026-03-02 00:07:16,1,latest_episode


In [19]:
#19 내부 테스트계정 300명 추가
# ── 테스트 계정 만들기 ──
# 실제 서비스에서 QA 담당자가 결제 테스트용으로 만든 계정을 흉내냄
# 특징: user_id가 test_로 시작, 가입 직후 노출 → 1분 만에 체험 → 5분 만에 결제, 코인 100개 한 번에 사용
N_TEST = 200
test_ids = [f"test_{i:04d}" for i in range(1, N_TEST + 1)]

# 노출 시각: 모집 기간 21일 중 아무 분(minute)
test_exposed = start + pd.to_timedelta(rng.integers(0, ENROLL_DAYS * 24 * 60, N_TEST), "min")
test_trial = test_exposed + pd.Timedelta(minutes=1)

# 테이블별 테스트 행 (모든 값이 똑같거나 비정상적으로 딱 떨어지게)
test_users = pd.DataFrame({
    "user_id": test_ids,
    "signup_date": (test_exposed.normalize() - pd.Timedelta(days=1)).date,   # 노출 하루 전 가입
    "age_group": "20대", "acquisition_channel": "organic", "preferred_genre": "로맨스판타지",
    "pre30_serial_reads": 0, "pre30_completed_reads": 0,
    "pre30_free_reads": 0, "pre30_coin_spend": 0,
})
test_exposures = pd.DataFrame({
    "user_id": test_ids, "variant": rng.choice(["A", "B"], N_TEST),
    "exposed_at": test_exposed, "platform": "app",
})
test_events = pd.concat([
    pd.DataFrame({"user_id": test_ids, "event_type": "trial_start",
                  "event_at": test_trial, "amount_krw": 0}),
    pd.DataFrame({"user_id": test_ids, "event_type": "payment",                # 체험 중인데 바로 결제
                  "event_at": test_trial + pd.Timedelta(minutes=5), "amount_krw": PASS_PRICE}),
])
test_coins = pd.DataFrame({
    "user_id": test_ids, "title_id": rng.choice(titles["title_id"], N_TEST),
    "txn_at": test_trial + pd.Timedelta(minutes=10), "coins": 100, "purpose": "latest_episode",
})
test_reading = pd.DataFrame({
    "user_id": test_ids, "read_date": test_trial.date, "access_type": "pass",
    "serial_reads": 0, "completed_reads": 50, "titles_finished": 0,
})

# ── 원래 테이블 뒤에 붙이기 ──
# 이벤트·코인 ID는 셀 21에서 새로 매기므로 기존 ID 컬럼은 지움
users_out = pd.concat([users, test_users], ignore_index=True)
exposures_out = pd.concat([exposures, test_exposures], ignore_index=True)
events_out = pd.concat([subscription_events.drop(columns="event_id"), test_events], ignore_index=True)
coins_out = pd.concat([coin_transactions.drop(columns="txn_id"), test_coins], ignore_index=True)
reading_out = pd.concat([daily_reading, test_reading], ignore_index=True)

In [20]:
# 품질이슈

# ── 이슈 1. 중복 결제 이벤트 (결제의 0.5%) ──
# 결제 행 중 일부를 골라서 똑같은 행을 한 번 더 붙임 (로그가 두 번 찍힌 상황)
pay_idx = events_out.index[events_out["event_type"] == "payment"]
dup_idx = rng.choice(pay_idx, int(len(pay_idx) * 0.005), replace=False)   # replace=False → 같은 행을 두 번 뽑지 않음
events_out = pd.concat([events_out, events_out.loc[dup_idx]], ignore_index=True)

# ── 이슈 2. 노출 기록 유실 (체험 시작자의 0.3%) ──
# 체험은 시작했는데 exposures에서 해당 유저 행을 지움 (노출 로그 누락 상황)
starters = t["user_id"].to_numpy()
lost = rng.choice(starters, int(len(starters) * 0.003), replace=False)
exposures_out = exposures_out[~exposures_out["user_id"].isin(lost)].copy()

# ── 이슈 3. 유입 경로 결측 (전체 유저의 2%) ──
missing = rng.random(len(users_out)) < 0.02
users_out.loc[missing, "acquisition_channel"] = np.nan    # np.nan = 빈 값

In [21]:
# ID 다시 매기기와 섞기 
# 중복·테스트 행이 끼어들었으므로 시간순으로 다시 정렬하고 ID를 새로 매김
# → 중복 행도 서로 다른 event_id를 갖게 되어, ID만 봐서는 중복을 못 찾게 됨 (정제 난이도 ↑)
events_out = events_out.sort_values(["event_at", "user_id"]).reset_index(drop=True)
events_out.insert(0, "event_id", [f"E{i:07d}" for i in range(1, len(events_out) + 1)])

coins_out = coins_out.sort_values(["txn_at", "user_id"]).reset_index(drop=True)
coins_out.insert(0, "txn_id", [f"C{i:07d}" for i in range(1, len(coins_out) + 1)])

# users, exposures는 행 순서를 무작위로 섞음
# → 테스트 계정이 맨 끝에 몰려 있으면 너무 쉽게 보이므로
# sample(frac=1) → 전체 행(100%)을 무작위 순서로 뽑기
users_out = users_out.sample(frac=1, random_state=SEED).reset_index(drop=True)
exposures_out = exposures_out.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [22]:
import os

# 노트북 위치(notebooks/) 기준 한 단계 위의 data 폴더
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)    # 폴더가 없으면 만들고, 있으면 그냥 넘어감

# 작은 테이블은 그냥 csv
titles.to_csv(f"{DATA_DIR}/titles.csv", index=False)                    # index=False → 0,1,2 행번호 열은 저장 안 함
events_out.to_csv(f"{DATA_DIR}/subscription_events.csv", index=False)

# 큰 테이블은 .csv.gz (파일명이 .gz로 끝나면 pandas가 알아서 압축)
users_out.to_csv(f"{DATA_DIR}/users.csv.gz", index=False)
exposures_out.to_csv(f"{DATA_DIR}/exposures.csv.gz", index=False)
coins_out.to_csv(f"{DATA_DIR}/coin_transactions.csv.gz", index=False)
reading_out.to_csv(f"{DATA_DIR}/daily_reading.csv.gz", index=False)

# 확인: 파일별 용량(MB)
for f in sorted(os.listdir(DATA_DIR)):
    size_mb = os.path.getsize(f"{DATA_DIR}/{f}") / 1024 / 1024
    print(f"{f:30s} {size_mb:5.1f} MB")

.ipynb_checkpoints               0.0 MB
coin_transactions.csv.gz        10.2 MB
daily_reading.csv.gz             6.0 MB
exposures.csv.gz                 2.3 MB
subscription_events.csv          3.0 MB
titles.csv                       0.2 MB
users.csv.gz                     4.0 MB
